In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])
print("ready", len(test_idx), "test contracts")

ready 153 test contracts


In [3]:
import random
from src.metrics import evaluate_ranker
from src.chunking import fixed_chunks


def random_ranker(contract, clause):
    chunks = fixed_chunks(contract["text"])
    random.Random(len(contract["text"])).shuffle(chunks)
    return chunks


systems = {"random": random_ranker, "dense_question": dense, "bm25": bm25, "hybrid_1_2": hybrid}
# optional "before" row, about 10-15 minutes on CPU:
# systems["dense_fixed"] = make_dense_ranker()

rows = {}
for name, r in systems.items():
    res = {cl: evaluate_ranker(r, contracts, test_idx, cl) for cl in TARGET_CLAUSES}
    df = pd.DataFrame(res).T
    df.to_csv(f"../eval/results_{name}_test.csv")
    rows[name] = df[["hit@1", "hit@3", "hit@10", "mrr"]].mean()
pd.DataFrame(rows).T.round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,hit@1,hit@3,hit@10,mrr
random,0.050,0.151,0.412,0.161
dense_question,0.590,0.787,0.919,0.704
bm25,0.722,0.887,0.953,0.810
hybrid_1_2,0.742,0.892,0.960,0.825


In [4]:
from src.abstain import load_thresholds
th = load_thresholds()
out = []
for cl in ["Governing Law", "Cap On Liability"]:
    t = th[cl]["threshold"]
    pos = neg = hp = hn = 0
    for i in test_idx:
        c = contracts[i]
        s = bm25(c, cl)[0]["score"]
        if c["gold"][cl]:
            pos += 1; hp += s >= t
        else:
            neg += 1; hn += s >= t
    out.append({"clause": cl, "present": pos, "kept": round(hp / pos, 2),
                "absent": neg, "false_alarm": round(hn / neg, 2)})
pd.DataFrame(out)

,clause,present,kept,absent,false_alarm
0,Governing Law,131,0.92,22,0.09
1,Cap On Liability,85,0.86,68,0.19
